In [1]:
import pandas as pd
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------------
# STEP 1: Load Dataset
# -----------------------------------
df = pd.read_csv("../data/complaints.csv")

print("Dataset Preview:")
print(df.head())

# -----------------------------------
# STEP 2: Define Features and Target
# -----------------------------------
X = df["complaint_text"]
y = df["department"]

# -----------------------------------
# STEP 3: Encode Labels
# -----------------------------------
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\nEncoded Departments:")
print(dict(zip(label_encoder.classes_,
               range(len(label_encoder.classes_)))))

# -----------------------------------
# STEP 4: Split Dataset (Fixed)
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.25,   # 5 test samples for 5 classes
    random_state=42,
    stratify=y_encoded
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# -----------------------------------
# STEP 5: TF-IDF Vectorization
# -----------------------------------
vectorizer = TfidfVectorizer(stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Shape:", X_train_tfidf.shape)

# -----------------------------------
# STEP 6: Train Better NLP Model
# -----------------------------------
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("\nModel trained successfully!")

# -----------------------------------
# STEP 7: Evaluate Model
# -----------------------------------
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# -----------------------------------
# STEP 8: Predict New Complaint
# -----------------------------------
sample_complaint = ["Garbage not collected for 3 days"]

sample_vector = vectorizer.transform(sample_complaint)

prediction = model.predict(sample_vector)

predicted_department = label_encoder.inverse_transform(prediction)

print("\nSample Complaint Prediction:")
print("Complaint:", sample_complaint[0])
print("Predicted Department:", predicted_department[0])

# -----------------------------------
# STEP 9: Save Model Files
# -----------------------------------
os.makedirs("../model", exist_ok=True)

with open("../model/complaint_classifier.pkl", "wb") as f:
    pickle.dump(model, f)

with open("../model/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("../model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("\nAll files saved successfully in model folder!")

Dataset Preview:
   complaint_id                              complaint_text department  \
0             1               No water supply in our street      Water   
1             2                Water leakage near main road      Water   
2             3                Low water pressure in colony      Water   
3             4  Drinking water not available since morning      Water   
4             5           Water pipeline broken near market      Water   

  location priority       status  
0   Zone A     High      Pending  
1   Zone B     High  In Progress  
2   Zone C   Medium      Pending  
3   Zone A     High      Pending  
4   Zone D     High     Resolved  

Encoded Departments:
{'Electricity': 0, 'Roads': 1, 'Sanitation': 2, 'Transport': 3, 'Water': 4}

Training Samples: 30
Testing Samples: 10

TF-IDF Shape: (30, 74)

Model trained successfully!

Model Accuracy: 0.7

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.50  